# 01 — Data Loading & Cleaning

**Home Energy Copilot** — CPSC 393 Final Project


1. Loads the RECS 2020 microdata file.
2. Drops the single household with non-positive `TOTALDOL`.
3. Merges the tiny `Subarctic` climate group into `Very-Cold` so peer groups stay statistically usable.
4. Derives per-end-use dollar columns (`heat_cool_dol`, `water_heat_dol`, `lighting_dol`, `fridge_dol`, `other_dol`) from per-fuel BTU shares so they reconcile to `TOTALDOL`.
5. Defines the efficiency class as a tertile of `TOTALDOL / TOTSQFT_EN` *within each climate peer group*.
6. Feature selection including dropping ids, leaky variables, variables with too many missing values, etc.
7. Feature shortlist (~55 vars) used by all downstream notebooks, seperated by group.
8. Documents RECS sentinel values (`-2` not applicable, `-9` refused) — they are **not** replaced with `NaN` here; downstream notebooks decide per-feature.
9. Saves a clean dataframe (`data/recs2020_clean.pkl`) and a metadata JSON for reproducibility.

## Setup

In [1]:
import os, json
import pandas as pd
import numpy as np

# Paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR     = os.path.join(PROJECT_ROOT, 'data')
RAW_CSV      = os.path.join(DATA_DIR, 'recs2020_public_v7.csv')
CLEAN_PKL    = os.path.join(DATA_DIR, 'recs2020_clean.pkl')
META_JSON    = os.path.join(DATA_DIR, 'recs2020_clean_meta.json')

# RECS sentinels: -2 = Not applicable, -9 = Refused / Don't know
SENTINELS = [-2, -9]
print('Project root:', PROJECT_ROOT)


Project root: /Users/aryakumar/Documents/Claude/Projects/CPSC393Final


## Step 1 — Load raw RECS 2020 microdata

In [2]:
df = pd.read_csv(RAW_CSV, low_memory=False)
print('shape:', df.shape)
print('memory MB:', round(df.memory_usage(deep=True).sum()/1e6, 1))
df.head(2)


shape: (18496, 799)
memory MB: 125.3


,DOEID,REGIONC,DIVISION,STATE_FIPS,state_postal,state_name,BA_climate,IECC_climate_code,UATYP10,HDD65,...,EVCHRGHOME,EVCHRGAPT,EVCHRGWKS,EVCHRGBUS,EVCHRGMUNI,EVCHRGDLR,EVCHRGHWY,EVCHRGOTH,EVHOMEAMT,EVCHRGTYPE
0,100001,WEST,Mountain South,35,NM,New Mexico,Mixed-Dry,4B,U,3844,...,-2.0,-2,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0
1,100002,SOUTH,West South Central,5,AR,Arkansas,Mixed-Humid,4A,U,3766,...,-2.0,-2,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0,-2.0


## Step 2 — Drop households with non-positive TOTALDOL

RECS reports `TOTALDOL` as the household's total annual energy expenditure. A small number of records can be ≤ 0 (e.g., net solar credits or a calculation edge case). They are not meaningful for our regression target, so we drop them.

In [3]:
n_before = len(df)
df = df[df['TOTALDOL'] > 0].copy()
print(f'dropped {n_before - len(df)} rows; remaining {len(df)}')


dropped 1 rows; remaining 18495


## Step 3 — Merge `Subarctic` into `Very-Cold`

RECS provides 8 Building America climate zones, but `Subarctic` only has 54 households — too few to support a within-region tertile cut. We merge it into `Very-Cold` (its closest analog) and store the result in `BA_climate_grp`. The original `BA_climate` is preserved.

In [4]:
df['BA_climate_grp'] = df['BA_climate'].replace({'Subarctic': 'Very-Cold'})
df['BA_climate_grp'].value_counts()


BA_climate_grp
Cold           7115
Mixed-Humid    5579
Hot-Humid      2545
Hot-Dry        1577
Marine          911
Very-Cold       626
Mixed-Dry       142
Name: count, dtype: int64

## Step 4 — Derive per-end-use dollar columns

RECS publishes per-fuel totals (`DOLLAREL`, `DOLLARNG`, `DOLLARLP`, `DOLLARFO`) and per-fuel-per-end-use BTU columns (e.g. `BTUELSPH`, `BTUNGWTH`, etc.). We derive end-use dollars by computing each row's per-fuel `$/BTU` rate and applying it to the relevant end-use BTU columns.

Notes on the column choice:

- `BTUELAHUHEAT` / `BTUELAHUCOL` are the air-handler fan portion of heating/cooling — they belong in `heat_cool_dol`.
- `BTUELRFG` already aggregates `BTUELRFG1 + BTUELRFG2`, so we use `BTUELRFG` only.
- `other_dol` is computed as the residual `TOTALDOL − sum(named buckets)`, clipped at 0. This guarantees totals reconcile and absorbs cooking, dishwashers, dryers, TVs, EV charging, pools, etc.

In [5]:
# Per-fuel rate ($/BTU) computed per row (preserves regional price variation).
fuels = {
    'EL': ('DOLLAREL', 'BTUEL'),
    'NG': ('DOLLARNG', 'BTUNG'),
    'LP': ('DOLLARLP', 'BTULP'),
    'FO': ('DOLLARFO', 'BTUFO'),
}
rates = {}
for f, (dol_col, btu_col) in fuels.items():
    if dol_col in df.columns and btu_col in df.columns:
        with np.errstate(divide='ignore', invalid='ignore'):
            rates[f] = np.where(df[btu_col] > 0, df[dol_col] / df[btu_col], 0.0)
        print(f'{f}: {(df[btu_col]>0).mean():.0%} of homes use this fuel')


EL: 100% of homes use this fuel
NG: 60% of homes use this fuel
LP: 11% of homes use this fuel
FO: 7% of homes use this fuel


In [6]:
# End-use BTU columns -> dollars
end_uses = {
    'heat':       ['BTUELSPH', 'BTUELAHUHEAT', 'BTUNGSPH', 'BTULPSPH', 'BTUFOSPH'],
    'cool':       ['BTUELCOL', 'BTUELAHUCOL'],
    'water_heat': ['BTUELWTH', 'BTUNGWTH', 'BTULPWTH', 'BTUFOWTH'],
    'lighting':   ['BTUELLGT'],
    'fridge':     ['BTUELRFG'],
}

def end_use_dollars(cols):
    total = pd.Series(0.0, index=df.index)
    for col in cols:
        if col not in df.columns:
            continue
        f = col[3:5]
        if f not in rates:
            continue
        total = total + df[col].clip(lower=0) * rates[f]
    return total

df['heat_cool_dol']  = end_use_dollars(end_uses['heat']) + end_use_dollars(end_uses['cool'])
df['water_heat_dol'] = end_use_dollars(end_uses['water_heat'])
df['lighting_dol']   = end_use_dollars(end_uses['lighting'])
df['fridge_dol']     = end_use_dollars(end_uses['fridge'])
named_dol = (df['heat_cool_dol'] + df['water_heat_dol']
             + df['lighting_dol'] + df['fridge_dol'])
df['other_dol'] = (df['TOTALDOL'] - named_dol).clip(lower=0)
df['derived_total_dol'] = named_dol + df['other_dol']

err = df['derived_total_dol'] - df['TOTALDOL']
print(f'reconciliation error: mean ${err.mean():.2f}, median ${err.median():.2f}, max abs ${err.abs().max():.2f}')
print()
shares = pd.Series({
    'heat_cool':  (df['heat_cool_dol']/df['TOTALDOL']).mean(),
    'water_heat': (df['water_heat_dol']/df['TOTALDOL']).mean(),
    'lighting':   (df['lighting_dol']/df['TOTALDOL']).mean(),
    'fridge':     (df['fridge_dol']/df['TOTALDOL']).mean(),
    'other':      (df['other_dol']/df['TOTALDOL']).mean(),
})
print('mean share of TOTALDOL by end-use bucket:')
print(shares.apply(lambda x: f'{x:.1%}'))


reconciliation error: mean $0.04, median $0.00, max abs $405.97

mean share of TOTALDOL by end-use bucket:
heat_cool     42.8%
water_heat    16.1%
lighting       4.8%
fridge         6.7%
other         29.5%
dtype: object


## Step 5 — Define the efficiency class

Per the project plan: classification target = tertile of `TOTALDOL / TOTSQFT_EN` (cost per square foot) **within each climate peer group**. This makes 'inefficient' a relative judgment vs. climatically similar homes, not vs. the entire U.S.

In [7]:
df['cost_per_sqft'] = df['TOTALDOL'] / df['TOTSQFT_EN']

def label_tertile(s):
    q1, q2 = s.quantile([1/3, 2/3])
    return pd.cut(s, [-np.inf, q1, q2, np.inf],
                  labels=['efficient', 'average', 'inefficient'])

df['efficiency_class'] = (
    df.groupby('BA_climate_grp', group_keys=False)['cost_per_sqft']
      .transform(lambda s: label_tertile(s))
)

print(df.groupby(['BA_climate_grp','efficiency_class'], observed=True).size().unstack())
print()
print('Overall counts:', df['efficiency_class'].value_counts().to_dict())


efficiency_class  efficient  average  inefficient
BA_climate_grp                                   
Cold                   2372     2371         2372
Hot-Dry                 526      525          526
Hot-Humid               848      849          848
Marine                  304      303          304
Mixed-Dry                48       47           47
Mixed-Humid            1860     1859         1860
Very-Cold               209      208          209

Overall counts: {'efficient': 6167, 'inefficient': 6166, 'average': 6162}


## Step 6 — Feature Selection

Pipeline:

1. **Drop ID columns.**
2. **Drop target-derived / leakage columns** — anything that is essentially `TOTALDOL` decomposed (`DOL*` per-fuel and per-end-use dollars, `BTU*`, `KWH*`, `CUFEET*`, `GALLON*`, `TOTAL*`, `*XBTU`, plus the per-end-use dollar columns we just derived in Step 4 and the targets themselves).
3. **Drop columns where > 50% of values are missing** after masking sentinels (`-2`, `-9`) to `NaN` — anything mostly "not applicable / refused" is too sparse for a Pearson view.
4. **Drop near-duplicate numeric features** using a |Pearson r| > 0.75 cutoff on the cleaned numeric block.
5. **Score each survivor with the right metric for its type.**
   - **Continuous-coded** features (cardinality > 20 unique values): `|Pearson r|` against `TOTALDOL` and `cost_per_sqft`.
   - **Categorical-coded** features (cardinality ≤ 20 unique values): **Cramér's V** against the categorical target `efficiency_class`. Cramér's V is the standard 0–1-bounded measure of association between two categorical variables, derived from the chi-square statistic of their contingency table.
6. **Drop features that score weakly** — drop if the appropriate metric is below `0.05` (i.e. `|Pearson r| < 0.05` for continuous, or `Cramér's V < 0.05` for categorical).
7. **Report top features** for each type separately: Pearson correlates against the dollar targets for continuous features, and Cramér's V against `efficiency_class` for categorical features.

In [16]:
import re
from scipy.stats import chi2_contingency

raw = df  # the raw 799-col frame already filtered to TOTALDOL > 0 with cost_per_sqft derived

ID_LIKE = {'DOEID'} | {c for c in raw.columns if re.fullmatch(r'NWEIGHT\d*', c)} \
                   | {c for c in raw.columns if c.startswith('BRRWGT')}

LEAKAGE_PREFIXES = ('DOL', 'BTU', 'KWH', 'CUFEET', 'GALLON', 'TOTAL')
LEAKAGE = {c for c in raw.columns if c.startswith(LEAKAGE_PREFIXES) or c.endswith('XBTU')}
LEAKAGE |= {'TOTALDOL', 'cost_per_sqft', 'efficiency_class',
            'heat_cool_dol', 'water_heat_dol', 'lighting_dol', 'fridge_dol', 'other_dol'}

numeric_all = raw.select_dtypes(include=np.number).columns.tolist()
candidate_cols = [c for c in numeric_all if c not in ID_LIKE and c not in LEAKAGE]


X = raw[candidate_cols].replace({s: np.nan for s in SENTINELS})

MISSING_THRESH = 0.50
missing_frac = X.isna().mean()
high_missing = missing_frac[missing_frac > MISSING_THRESH].sort_values(ascending=False)
X = X.drop(columns=high_missing.index)
candidate_cols = X.columns.tolist()


corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape, dtype=bool), k=1))

DUP_THRESH = 0.75
to_drop = set()
for col in upper.columns:
    high_partners = upper.index[upper[col] > DUP_THRESH]
    for partner in high_partners:
        loser = sorted([col, partner])[1]
        to_drop.add(loser)

kept = [c for c in candidate_cols if c not in to_drop]

N_UNIQUE_CAT = 20
cat_cols = [c for c in kept if X[c].dropna().nunique() <= N_UNIQUE_CAT]
con_cols = [c for c in kept if c not in cat_cols]

y_dol = raw['TOTALDOL']
y_per = raw['cost_per_sqft']
y_cls = raw['efficiency_class']

r_dol = X[con_cols].corrwith(y_dol).abs()
r_per = X[con_cols].corrwith(y_per).abs()
strength_con = pd.concat([r_dol, r_per], axis=1).max(axis=1)

def cramers_v(x, y):
    """Cramér's V between two categorical-coded series; NaN-tolerant, bounded [0, 1]."""
    mask = x.notna() & y.notna()
    if mask.sum() < 2:
        return np.nan
    ct = pd.crosstab(x[mask], y[mask])
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return 0.0
    chi2 = chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    k = min(ct.shape) - 1
    return float(np.sqrt(chi2 / (n * k))) if k > 0 else 0.0

strength_cat = pd.Series({c: cramers_v(X[c], y_cls) for c in cat_cols}, dtype=float)
strength = pd.concat([strength_con, strength_cat]).dropna()

WEAK_THRESH = 0.05
weak = strength[strength < WEAK_THRESH].index.tolist()
kept = [c for c in kept if c not in weak]


final_con = [c for c in kept if c in con_cols]
final_cat = [c for c in kept if c in cat_cols]

pearson_dol = X[final_con].corrwith(y_dol).dropna().sort_values()
pearson_per = X[final_con].corrwith(y_per).dropna().sort_values()
cramers_eff = strength_cat.loc[final_cat].sort_values()



## Step 7 — Freeze the feature shortlist

RECS has ~780 columns; we use ~55 grouped by theme. This shortlist is the input to all downstream models.

In [8]:
features = {
    'climate':   ['BA_climate_grp','HDD65','CDD65','REGIONC','DIVISION','state_postal'],
    'building':  ['TYPEHUQ','YEARMADERANGE','BEDROOMS','TOTROOMS','NCOMBATH','NHAFBATH',
                  'STORIES','TOTSQFT_EN','WALLTYPE','ROOFTYPE','HIGHCEIL'],
    'envelope':  ['ADQINSUL','DRAFTY','TYPEGLASS','WINDOWS','ORIGWIN','WINFRAME','TREESHAD'],
    'hvac':      ['EQUIPM','FUELHEAT','EQUIPAGE','AIRCOND','TYPETHERM',
                  'TEMPHOME','TEMPGONE','TEMPNITE'],
    'water':     ['FUELH2O','WHEATAGE','WHEATSIZ'],
    'appliance': ['NUMFRIG','SIZRFRI1','TYPERFR1','AGERFRI1','RANGEFUEL',
                  'CWASHER','TOPFRONT','WASHTEMP','AGECWASH','DISHWASH','AGEDW'],
    'lighting':  ['LGTIN1TO4','LGTIN4TO8','LGTINMORE8','LGTINLED','LGTINCFL','LGTINCAN'],
    'occupants': ['NHSLDMEM','HHAGE','EMPLOYHH'],
}
all_features = sorted({c for grp in features.values() for c in grp})
present = [c for c in all_features if c in df.columns]
missing = [c for c in all_features if c not in df.columns]
print(f'features present: {len(present)} / {len(all_features)}')
if missing:
    print('MISSING:', missing)


features present: 55 / 55


## Step 8 — Sentinel-value scan

RECS sentinels (`-2` Not applicable, `-9` Refused) are NOT replaced with `NaN` here because some are meaningful (e.g. `EQUIPM=-2` literally means 'no heating equipment installed'). Downstream notebooks decide per-feature.

In [9]:
sent_counts = {}
for c in present:
    if pd.api.types.is_numeric_dtype(df[c]):
        n = df[c].isin(SENTINELS).sum()
        if n > 0:
            sent_counts[c] = int(n)
print(f'{len(sent_counts)} feature columns contain at least one sentinel value:')
for k, v in sorted(sent_counts.items(), key=lambda x: -x[1]):
    print(f'  {k:20s} {v:>6d} sentinel rows')


18 feature columns contain at least one sentinel value:
  STORIES                4426 sentinel rows
  AGEDW                  4388 sentinel rows
  ROOFTYPE               2439 sentinel rows
  AGECWASH               2152 sentinel rows
  TOPFRONT               2152 sentinel rows
  WASHTEMP               2152 sentinel rows
  RANGEFUEL              2067 sentinel rows
  HIGHCEIL                974 sentinel rows
  EQUIPAGE                751 sentinel rows
  EQUIPM                  751 sentinel rows
  FUELHEAT                751 sentinel rows
  TEMPGONE                751 sentinel rows
  TEMPHOME                751 sentinel rows
  TEMPNITE                751 sentinel rows
  TYPETHERM               218 sentinel rows
  TYPERFR1                211 sentinel rows
  AGERFRI1                 83 sentinel rows
  SIZRFRI1                 83 sentinel rows


## Step 9 — Save processed dataset and metadata

In [10]:
keep_cols = list(dict.fromkeys(
    ['DOEID','NWEIGHT','BA_climate','BA_climate_grp','TOTALDOL','TOTSQFT_EN',
     'cost_per_sqft','efficiency_class',
     'heat_cool_dol','water_heat_dol','lighting_dol','fridge_dol','other_dol',
     'TOTALBTU','BTUEL','BTUNG'] + present
))
keep_cols = [c for c in keep_cols if c in df.columns]
clean = df[keep_cols].copy()

try:
    clean.to_parquet(CLEAN_PKL.replace('.pkl', '.parquet'))
    save_path = CLEAN_PKL.replace('.pkl', '.parquet')
except Exception:
    clean.to_pickle(CLEAN_PKL)
    save_path = CLEAN_PKL

meta = {
    'rows': int(len(clean)),
    'cols': int(clean.shape[1]),
    'feature_groups': features,
    'all_features_present': present,
    'features_missing': missing,
    'efficiency_class_counts': clean['efficiency_class'].value_counts().to_dict(),
    'ba_climate_grp_counts': clean['BA_climate_grp'].value_counts().to_dict(),
    'sentinel_value_codes': SENTINELS,
    'columns_with_sentinels': sent_counts,
    'notes': [
        'Dropped 1 row with TOTALDOL <= 0.',
        'Subarctic merged into Very-Cold for peer-group sizing.',
        'Sentinel values not replaced with NaN; downstream handles per-feature.',
        'efficiency_class = tertile of TOTALDOL/TOTSQFT_EN within BA_climate_grp.',
        'Per-end-use dollar columns derived from per-fuel BTU times row-level $/BTU rates; '
        'other_dol is the residual to TOTALDOL.',
    ],
}
with open(META_JSON, 'w') as f:
    json.dump(meta, f, indent=2, default=str)

print('saved:', save_path)
print('saved:', META_JSON)
print('clean shape:', clean.shape)


saved: /Users/aryakumar/Documents/Claude/Projects/CPSC393Final/data/recs2020_clean.parquet
saved: /Users/aryakumar/Documents/Claude/Projects/CPSC393Final/data/recs2020_clean_meta.json
clean shape: (18495, 69)


## Summary

- **Final shape:** `(18495, 69)` — 18,495 households, 69 columns (targets, weights, derived end-use dollars, and the 55-feature shortlist).
- **Class balance:** efficient / average / inefficient are equal-sized within each climate peer group by construction.
- **Reconciliation:** derived end-use dollars sum to `TOTALDOL` to within rounding error.